# Symptom Extraction & Classification Experiments

This notebook implements and evaluates three approaches:
1. **Regex/Keyword Matching**
2. **Zero-shot LLM**
3. **Chain-of-Thought (CoT) LLM**

## Objectives
1. **Symptom Extraction**: Predict trinary status (0=Not Present, 1=Negated, 2=Present) for all symptoms.
2. **Category Classification**: Predict the primary category (RES, CAR, MSK, GAS, DER) for each case.
3. **Reliability**: Run LLM experiments over 3 trials to calculate error bars.

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import time
import json
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# --- CONFIGURATION ---
GROUND_TRUTH_FILE = "gold_standard_trinary.csv"

# !!! ENTER YOUR API KEY HERE !!!
import os
OPENAI_API_KEY = "sk-..." # <--- REPLACE THIS WITH YOUR ACTUAL KEY

# Set to True to run LLM calls
RUN_LLM_EXPERIMENTS = False 
LLM_MODEL = "gpt-4o"
NUM_TRIALS = 3  # Number of runs for LLM to get error bars

# Load Data
if os.path.exists(GROUND_TRUTH_FILE):
    df_gt = pd.read_csv(GROUND_TRUTH_FILE)
    print(f"Loaded {len(df_gt)} cases from {GROUND_TRUTH_FILE}.")
else:
    print(f"Error: {GROUND_TRUTH_FILE} not found.")
    df_gt = pd.DataFrame()

# Updated Category Map including 'Other' symptoms
CATEGORY_MAP = {
    'RES': ['Fever', 'Cough', 'Dyspnea', 'Wheezing', 'Rhinorrhea', 'Pharyngitis', 'Congestion'],
    'CAR': ['Chest Pain', 'Palpitations', 'Syncope', 'Edema'],
    'MSK': ['Myalgia', 'Arthralgia', 'Back Pain'],
    'GAS': ['Nausea/Emesis', 'Diarrhea', 'Abdominal Pain'],
    'DER': ['Pruritus', 'Rash'],
    'OTH': ['Headache', 'Exposure To Sick', 'Anosmia', 'Ageusia', 'Fatigue']
}

ALL_SYMPTOMS = []
for cat_list in CATEGORY_MAP.values():
    ALL_SYMPTOMS.extend(cat_list)

# Verify these columns exist in the dataframe
MISSING_COLS = [s for s in ALL_SYMPTOMS if s not in df_gt.columns]
if MISSING_COLS:
    print(f"Warning: The following symptoms are in the map but NOT in the CSV: {MISSING_COLS}")
    ALL_SYMPTOMS = [s for s in ALL_SYMPTOMS if s in df_gt.columns]

print(f"Experimenting with {len(ALL_SYMPTOMS)} symptoms: {ALL_SYMPTOMS}")

def read_transcript(path):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        return ""

## Approach 1: Regex/Keyword Matching

In [ ]:
# Keywords and Negation Patterns
KEYWORDS = {
    'Fever': [r'fever', r'chills', r'night sweats', r'burning up', r'hot', r'temperature'],
    'Cough': [r'cough', r'hacking', r'barking'],
    'Dyspnea': [r'shortness of breath', r'breathless', r'difficulty breathing', r'winded', r'dyspnea', r'air hunger'],
    'Wheezing': [r'wheez', r'whistling'],
    'Rhinorrhea': [r'runny nose', r'nasal drip', r'congestion', r'stuffy nose'],
    'Pharyngitis': [r'sore throat', r'scratchy throat', r'throat pain'],
    'Chest Pain': [r'chest pain', r'chest tightness', r'pressure on chest', r'elephant on chest', r'chest discomfort'],
    'Palpitations': [r'palpitations', r'racing heart', r'skipped beats', r'thumping', r'heart flutter'],
    'Syncope': [r'faint', r'black out', r'pass out', r'syncope', r'lightheaded'],
    'Edema': [r'swollen', r'swelling', r'edema', r'puffy'],
    'Myalgia': [r'muscle ache', r'body ache', r'muscle pain', r'myalgia'],
    'Arthralgia': [r'joint pain', r'stiff joints', r'aching joints', r'arthralgia'],
    'Back Pain': [r'back pain', r'lumbar', r'lower back'],
    'Nausea/Emesis': [r'nausea', r'vomit', r'throw up', r'sick to stomach', r'queasy'],
    'Diarrhea': [r'diarrhea', r'loose stool', r'runs'],
    'Abdominal Pain': [r'stomach pain', r'belly ache', r'cramps', r'abdominal'],
    'Pruritus': [r'itch', r'scratching'],
    'Rash': [r'rash', r'hives', r'redness', r'breakout', r'lesion'],
    'Headache': [r'headache', r'migraine', r'head pounding'],
    'Exposure To Sick': [r'exposure', r'sick contact', r'someone sick', r'family sick', r'friend sick'],
    'Anosmia': [r'loss of smell', r'can\'t smell', r'anosmia'],
    'Ageusia': [r'loss of taste', r'can\'t taste', r'ageusia'],
    'Fatigue': [r'fatigue', r'tired', r'exhausted', r'lethargic', r'weak'],
    'Congestion': [r'congestion', r'congested', r'stuffed up']
}

NEGATIONS = [r'no ', r'not ', r'deny', r'denies', r'without ', r'free of ', r'never ']

def regex_extract_symptoms(text):
    """Returns a dict of {symptom: score}"""
    text = text.lower()
    results = {s: 0 for s in ALL_SYMPTOMS}
    
    for symptom, patterns in KEYWORDS.items():
        if symptom not in ALL_SYMPTOMS: continue # Skip if not in target list
        for pattern in patterns:
            matches = list(re.finditer(pattern, text))
            if matches:
                # Default to Present (2)
                current_score = 2
                # Check negation in window before match
                for match in matches:
                    start = max(0, match.start() - 30)
                    context = text[start:match.start()]
                    for neg in NEGATIONS:
                        if re.search(neg, context):
                            current_score = 1 # Negated
                            break
                    if current_score == 1:
                        break
                
                results[symptom] = max(results[symptom], current_score)
                
    return results

def regex_predict_category(symptom_scores):
    """Predicts category based on symptom scores (majority vote of Present/Negated)"""
    cat_scores = {c: 0 for c in CATEGORY_MAP if c != 'OTH'}
    for cat, symptoms in CATEGORY_MAP.items():
        if cat == 'OTH': continue # Don't vote for Other category
        for s in symptoms:
            # Give weight to Present (2) and Negated (1)
            if s in symptom_scores:
                if symptom_scores[s] == 2:
                    cat_scores[cat] += 2
                elif symptom_scores[s] == 1:
                    cat_scores[cat] += 1
    
    # Return category with max score
    return max(cat_scores, key=cat_scores.get)

## Approaches 2 & 3: LLM Wrappers (Multi-Trial)

In [ ]:
from openai import OpenAI
import json

client = None
if RUN_LLM_EXPERIMENTS:
    client = OpenAI(api_key=OPENAI_API_KEY)

def call_llm_json(prompt, system_prompt, model=LLM_MODEL):
    if not RUN_LLM_EXPERIMENTS or client is None:
        return "{}"
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ],
            response_format={ "type": "json_object" }, 
            temperature=0.5 # Slight temperature to allow variation across trials
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"LLM Call Failed: {e}")
        return "{}"

def parse_llm_json(response, symptoms_list):
    try:
        data = json.loads(response)
        preds = {s: 0 for s in symptoms_list}
        raw_scores = data.get("symptoms", {})
        for s, score in raw_scores.items():
            if s in preds:
                preds[s] = int(score)
        category = data.get("category", "Unknown")
        return preds, category, response
    except json.JSONDecodeError:
        return {s: 0 for s in symptoms_list}, "Unknown", response

In [ ]:
# --- PROMPTS ---

def get_zeroshot_json_prompts(transcript):
    sys = "You are a medical assistant expert in symptom extraction. You output strictly in JSON."
    symptom_list = json.dumps(ALL_SYMPTOMS)
    user = f'''
Analyze the transcript and identify the presence of these symptoms: {symptom_list}.

Return a JSON object with two keys:
1. "symptoms": detailed object where keys are symptom names and values are 0 (Not Mentioned), 1 (Negated), 2 (Present).
2. "category": one of ["RES", "CAR", "MSK", "GAS", "DER"]

Transcript:
"""{transcript}"""
'''
    return sys, user

def get_cot_json_prompts(transcript):
    sys = "You are a medical scribe. You think step-by-step but output the final result in strict JSON."
    symptom_list = json.dumps(ALL_SYMPTOMS)
    user = f'''
Step 1: Analyze the transcript for each of these symptoms: {symptom_list}.
Step 2: Reason about their presence using the scale:
  - 0: Not Mentioned / Not Present
  - 1: Negated (Explicitly mentioned as absent)
  - 2: Present (Explicitly mentioned as present)

Step 3: Determine the clinical category (RES, CAR, MSK, GAS, DER).

Provide your final answer as a VALID JSON object with this exact structure:
{{
  "reasoning": "Summary of your analysis...",
  "symptoms": {{
    "Fever": 2,
    "Cough": 0,
    ... for all symptoms (using scores 0, 1, or 2)
  }},
  "category": "RES"
}}

Transcript:
"""{transcript}"""
'''
    return sys, user

## Running Experiments (Loop Over Trials)
We will run `NUM_TRIALS` iterations for Zero-Shot and CoT.

In [ ]:
# Store Results: dict of lists to handle trials
# Structure: results_by_trial = { 0: [...], 1: [...], 2: [...] }
results_by_trial = {i: [] for i in range(NUM_TRIALS)}

print("Starting Experiments...")
print(f"LLM Enabled: {RUN_LLM_EXPERIMENTS}")
print(f"Trials: {NUM_TRIALS}")

for trial_idx in range(NUM_TRIALS):
    print(f"\n--- Starting Trial {trial_idx + 1}/{NUM_TRIALS} ---")
    
    trial_results = []
    
    for idx, row in df_gt.iterrows():
        fname = row['filename']
        transcript = read_transcript(row['full_path'])
        if not transcript: continue
            
        true_symptoms = {s: row[s] for s in ALL_SYMPTOMS}
        true_cat = row['category']
        
        # Regex (Run once, deterministic, but we copy it per trial for data consistency)
        regex_symptoms = regex_extract_symptoms(transcript)
        regex_cat = regex_predict_category(regex_symptoms)
        
        # Defaut values
        zs_symptoms, zs_cat = {s:0 for s in ALL_SYMPTOMS}, "Unknown"
        cot_symptoms, cot_cat = {s:0 for s in ALL_SYMPTOMS}, "Unknown"

        if RUN_LLM_EXPERIMENTS:
            # Rate limit pause
            time.sleep(0.5)
            
            # ZS
            sys_p, user_p = get_zeroshot_json_prompts(transcript)
            zs_resp = call_llm_json(user_p, sys_p)
            zs_symptoms, zs_cat, _ = parse_llm_json(zs_resp, ALL_SYMPTOMS)
            
            # CoT
            sys_p, user_p = get_cot_json_prompts(transcript)
            cot_resp = call_llm_json(user_p, sys_p)
            cot_symptoms, cot_cat, _ = parse_llm_json(cot_resp, ALL_SYMPTOMS)
        
        trial_results.append({
            'filename': fname,
            'true_symptoms': true_symptoms,
            'true_cat': true_cat,
            'regex_symptoms': regex_symptoms,
            'regex_cat': regex_cat,
            'zs_symptoms': zs_symptoms,
            'zs_cat': zs_cat,
            'cot_symptoms': cot_symptoms,
            'cot_cat': cot_cat
        })
        
    results_by_trial[trial_idx] = trial_results

print("All Trials Complete.")

## Visualization with Error Bars

In [ ]:
def flatten_results(results_list, method_prefix):
    y_true = []
    y_pred = []
    for res in results_list:
        true_map = res['true_symptoms']
        pred_map = res[f'{method_prefix}_symptoms']
        for s in ALL_SYMPTOMS:
            if s in true_map and s in pred_map:
                y_true.append(true_map[s])
                y_pred.append(pred_map[s])
    return y_true, y_pred

# Collect Stats per Method per Trial
methods = ['regex', 'zs', 'cot']
symptom_stats = {m: {'f1_scores': [], 'acc_scores': []} for m in methods}
category_stats = {m: {'acc_scores': []} for m in methods}

if RUN_LLM_EXPERIMENTS:
    runs = range(NUM_TRIALS)
else:
    runs = [0] 

for t in runs:
    res_list = results_by_trial[t]
    for m in methods:
        if m != 'regex' and not RUN_LLM_EXPERIMENTS:
            continue
            
        # 1. Symptom Stats
        y_t, y_p = flatten_results(res_list, m)
        if y_t:
            f1 = f1_score(y_t, y_p, average='macro')
            acc = accuracy_score(y_t, y_p)
            symptom_stats[m]['f1_scores'].append(f1)
            symptom_stats[m]['acc_scores'].append(acc)
            
        # 2. Category Stats
        cat_true = [r['true_cat'] for r in res_list]
        cat_pred = [r[f'{m}_cat'] for r in res_list]
        if cat_true:
            cat_acc = accuracy_score(cat_true, cat_pred)
            category_stats[m]['acc_scores'].append(cat_acc)

# Check if stats are empty
methods_to_plot = [m for m in methods if symptom_stats[m]['f1_scores']]

if not methods_to_plot:
    print("No results to plot. Did you enable RUN_LLM_EXPERIMENTS?")
else:
    sns.set_context("poster")
    sns.set_style("whitegrid")
    colors = sns.color_palette("viridis", len(methods_to_plot))
    
    # --- PLOT 1: SYMPTOM EXTRACTION ---
    fig, ax = plt.subplots(1, 2, figsize=(18, 7))
    
    x_labels = [m.upper() for m in methods_to_plot]
    f1_means = [np.mean(symptom_stats[m]['f1_scores']) for m in methods_to_plot]
    f1_stds = [np.std(symptom_stats[m]['f1_scores']) for m in methods_to_plot]
    
    acc_means = [np.mean(symptom_stats[m]['acc_scores']) for m in methods_to_plot]
    acc_stds = [np.std(symptom_stats[m]['acc_scores']) for m in methods_to_plot]

    # F1 Plot
    bars1 = ax[0].bar(x_labels, f1_means, yerr=f1_stds, capsize=10, color=colors, edgecolor='black', alpha=0.9)
    ax[0].set_title('Symptom Extraction: Macro F1', fontsize=20, pad=20)
    ax[0].set_ylim(0, 1.05)
    ax[0].bar_label(bars1, fmt='%.2f', padding=5, fontsize=14)
    
    # Accuracy Plot
    bars2 = ax[1].bar(x_labels, acc_means, yerr=acc_stds, capsize=10, color=colors, edgecolor='black', alpha=0.9)
    ax[1].set_title('Symptom Extraction: Accuracy', fontsize=20, pad=20)
    ax[1].set_ylim(0, 1.05)
    ax[1].bar_label(bars2, fmt='%.2f', padding=5, fontsize=14)
    
    plt.suptitle("Task 1: Symptom Extraction Performance", fontsize=24, y=1.05)
    plt.tight_layout()
    plt.show()

    # --- PLOT 2: CATEGORY CLASSIFICATION ---
    fig2, ax2 = plt.subplots(figsize=(10, 7))
    
    cat_acc_means = [np.mean(category_stats[m]['acc_scores']) for m in methods_to_plot]
    cat_acc_stds = [np.std(category_stats[m]['acc_scores']) for m in methods_to_plot]
    
    bars3 = ax2.bar(x_labels, cat_acc_means, yerr=cat_acc_stds, capsize=10, color=colors, edgecolor='black', alpha=0.9)
    ax2.set_title('Task 2: Category Classification Accuracy', fontsize=20, pad=20)
    ax2.set_ylim(0, 1.05)
    ax2.bar_label(bars3, fmt='%.2f', padding=5, fontsize=14)
    ax2.set_ylabel("Accuracy")
    
    plt.tight_layout()
    plt.show()

    # --- DETAILED REPORTS ---
    print("\n" + "="*60)
    print("DETAILED METRICS (Last Trial)")
    print("="*60 + "\n")
    
    last_trial_idx = runs[-1]
    last_res = results_by_trial[last_trial_idx]
    
    for m in methods_to_plot:
        print(f"\n>> Method: {m.upper()}")
        print("-" * 30)
        
        # 1. Symptom Report
        y_t, y_p = flatten_results(last_res, m)
        print("Symptom Extraction Report:")
        print(classification_report(y_t, y_p, target_names=['Not Present (0)', 'Negated (1)', 'Present (2)']))
        
        # 2. Category Report
        cat_true = [r['true_cat'] for r in last_res]
        cat_pred = [r[f'{m}_cat'] for r in last_res]
        
        if cat_true:
            print("Category Classification Report:")
            # Get unique labels sorted to ensure consistent matrix
            labels = sorted(list(set(cat_true + cat_pred)))
            print(classification_report(cat_true, cat_pred, zero_division=0))
            
            # Category Confusion Matrix
            cm = confusion_matrix(cat_true, cat_pred, labels=labels)
            plt.figure(figsize=(8, 6))
            sns.set_context("notebook")
            sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', 
                        xticklabels=labels, yticklabels=labels)
            plt.title(f'Category Confusion Matrix: {m.upper()}')
            plt.ylabel('True Category')
            plt.xlabel('Predicted Category')
            plt.show()
            
    # --- 3. Per-Symptom Performance Heatmap ---
    print("\n" + "="*60)
    print("PER-SYMPTOM F1 SCORES (Last Trial)")
    print("="*60 + "\n")
    
    symptom_data = [] 
    
    for m in methods_to_plot:
        res_list = results_by_trial[runs[-1]]
        
        for s in ALL_SYMPTOMS:
            y_true_s = [r['true_symptoms'].get(s, 0) for r in res_list]
            y_pred_s = [r[f'{m}_symptoms'].get(s, 0) for r in res_list]
            
            f1_s = f1_score(y_true_s, y_pred_s, average='macro', zero_division=0)
            symptom_data.append({
                'Method': m.upper(),
                'Symptom': s,
                'F1 Score': f1_s
            })
    
    df_symptom = pd.DataFrame(symptom_data)
    heatmap_data = df_symptom.pivot(index="Symptom", columns="Method", values="F1 Score")
    
    heatmap_data['Mean'] = heatmap_data.mean(axis=1)
    heatmap_data = heatmap_data.sort_values('Mean', ascending=False).drop(columns=['Mean'])

    # Split into 2 columns to make it less tall
    n_symptoms = len(heatmap_data)
    mid = (n_symptoms + 1) // 2
    
    data_left = heatmap_data.iloc[:mid]
    data_right = heatmap_data.iloc[mid:]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7)) 
    
    sns.set_context("notebook")
    sns.heatmap(data_left, annot=True, cmap="YlGnBu", fmt=".2f", linewidths=.5, ax=ax1, cbar=False)
    ax1.set_title("Per-Symptom F1 (Top Half)", fontsize=14)
    
    sns.heatmap(data_right, annot=True, cmap="YlGnBu", fmt=".2f", linewidths=.5, ax=ax2)
    ax2.set_title("Per-Symptom F1 (Bottom Half)", fontsize=14)
    ax2.set_ylabel("") 
    
    plt.tight_layout()
    plt.show()